# Cleaning the Datasets
### Import necessary packages

In [1]:
import pandas as pd
import re
import os

### Create `normalize_title` function to standardize titles across datasets

In [2]:
def normalize_title(title):
    """
    Consistent title formatting used across ALL datasets for joining
    ex. "Spider-Man: No Way Home" becomes "spiderman no way home"
    """
    title = str(title).lower()
    title = re.sub(r"[^\w\s]", "", title)  # removes all punctuation
    title = re.sub(r"\s+", " ", title)      # removes extra whitespace
    return title.strip()

### IMDb Basics Dataset

In [3]:
# Load only relevant columns, treat \N as missing
imdb_basics = pd.read_csv("../data/raw/title.basics.tsv", sep="\t",
    usecols=["tconst", "titleType", "primaryTitle", "startYear", "runtimeMinutes", "genres"],
    dtype=str,
    na_values="\\N")

# Drop all rows with any missing values
imdb_basics = imdb_basics.dropna()

# Keep only movies
imdb_basics = imdb_basics[imdb_basics["titleType"] == "movie"]

# Convert startYear to numeric, drop the invalid conversions
imdb_basics["startYear"] = pd.to_numeric(imdb_basics["startYear"], errors = "coerce")
imdb_basics = imdb_basics.dropna(subset = ["startYear"])

# Filter for movies between 2010–2024
imdb_basics = imdb_basics[imdb_basics["startYear"].between(2010, 2024)]

# Normalize movie titles
imdb_basics["norm_title"] = imdb_basics["primaryTitle"].apply(normalize_title)

# Keep only relevant columns
imdb_basics = imdb_basics[["norm_title", "tconst", "titleType", "startYear", "runtimeMinutes", "genres"]]

print(f"Movies in IMDb basics (2010–2024): {len(imdb_basics)}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/title.basics.tsv'

### IMDb Ratings Dataset

In [29]:
#Load IMDb Ratings dataset
imdb_ratings = pd.read_csv("../data/raw/title.ratings.tsv", sep="\t",
    na_values="\\N")

imdb_ratings = imdb_ratings[["tconst", "averageRating"]] #drop numVotes

print(f"Ratings loaded: {len(imdb_ratings)}")

Ratings loaded: 1647633


In [30]:
imdb_ratings.head()

,tconst,averageRating
0,tt0000001,5.7
1,tt0000002,5.5
2,tt0000003,6.4
3,tt0000004,5.1
4,tt0000005,6.2


In [37]:
# Merge IMDb basics with ratings
movies = pd.merge(imdb_basics, imdb_ratings, on="tconst", how="left")

# Drop rows with missing ratings
movies = movies.dropna(subset=["averageRating"])

# Convert averageRating to numeric
movies["averageRating"] = pd.to_numeric(movies["averageRating"], errors="coerce")

# Drop rows where conversion failed
movies = movies.dropna(subset=["averageRating"])

# Select 2000 random movies
movies = movies.sample(n=2000, random_state=42)

print(f"Random 2000 movies selected: {len(movies)}")

Random 2000 movies selected: 2000


### Box Office Dataset

In [32]:
# Load box office dataset
boxoffice = pd.read_csv("../data/raw/enhanced_box_office_data(2000-2024)u.csv")

boxoffice = boxoffice.dropna(subset=["$Worldwide"]) # Drops rows missing values for worldwide revenue
boxoffice = boxoffice.drop_duplicates() #Removes duplicate rows

boxoffice["norm_title"] = boxoffice["Release Group"].apply(normalize_title)

boxoffice = boxoffice[["norm_title", "Genres", "Year", "$Worldwide"]]

# Select 2000 box office movies randomly
boxoffice = boxoffice.sample(n=2000, random_state=42)

print(f"Box Office loaded: {len(boxoffice)}")

Box Office loaded: 2000


In [33]:
boxoffice.head()

,norm_title,Genres,Year,$Worldwide
1501,we own the night,"Drama, Crime, Thriller",2007,55033767.0
2586,i want you,"Drama, Romance",2012,23961214.0
2653,escape plan,"Action, Thriller",2013,137328301.0
1055,red eye,"Thriller, Mystery",2005,96258201.0
705,dumb and dumberer when harry met lloyd,Comedy,2003,39267515.0


### Rotten Tomatoes Dataset

In [34]:
# Load Rotten Tomatoes dataset; drop rows missing consensus or ratings
rotten = pd.read_csv("../data/raw/rotten_tomatoes_movies.csv",
    usecols=["movie_title", "critics_consensus", "tomatometer_rating"])

rotten = rotten.dropna(subset=["critics_consensus", "tomatometer_rating"])
rotten["norm_title"] = rotten["movie_title"].apply(normalize_title)

rotten = rotten[["norm_title", "critics_consensus", "tomatometer_rating"]] #drop movie_title, use norm_title

# Select 2000 rotten tomatoes reviews randomly
rotten = rotten.sample(n=2000, random_state=42)

print(f"Rotton Tomatoes loaded: {len(rotten)}")

Rotton Tomatoes loaded: 2000


In [35]:
rotten.head()

,norm_title,critics_consensus,tomatometer_rating
1415,spellbound,"A suspenseful, gripping documentary that featu...",97.0
86,molière,"Moliere is a sophisticated, witty biopic of th...",69.0
7833,home on the range,Though Home on the Range is likeable and may k...,53.0
2939,ash is purest white,Ash Is Purest White finds writer-director Zhan...,99.0
7241,gran torino,Though a minor entry in Eastwood's body of wor...,81.0


The column names of all dataframe are printed to select the one that are relevant to our database design and research questions. These chosen columns will become the attributes in our SQL tables.

### Save all cleaned datasets to CSV file

In [38]:
movies.to_csv("../data/clean/movies_clean.csv", index=False)
boxoffice.to_csv("../data/clean/boxoffice_clean.csv", index=False)
rotten.to_csv("../data/clean/rotten_clean.csv", index=False)